In [ ]:
from collections import defaultdict
from dataclasses import dataclass
from io import StringIO
from pathlib import Path
import re

import numpy as np
import pandas as pd
import polars as pl
import polars.lazyframe as plf

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)


In [ ]:
dataset_fps = list(RAW_DIR.glob('*.parquet'))

df = pl.concat([pl.scan_parquet(fp) for fp in dataset_fps[:2]])
df.head().count().collect(), df.head().collect()


In [ ]:
import re
import pymorphy3

def clean_and_normalize_russian_text_advanced(text: str) -> list:
    # 1. Initialize the Morphological Analyzer
    morph = pymorphy3.MorphAnalyzer()
    
    # 2. Convert to lowercase
    text_lowercase = text.lower()
    
    # 3. Advanced Tokenization via Regex
    # Pattern breakdown:
    # [а-яё]+       -> Matches one or more Russian letters
    # (?:-[а-яё]+)* -> Matches an optional internal hyphen followed by more Russian letters
    #                  The '*' allows multiple hyphens (e.g., "рок-н-ролл")
    word_pattern = r'[а-яё]+(?:-[а-яё]+)*'
    words = re.findall(word_pattern, text_lowercase)
    
    cleaned_words = []
    
    # 4. Filter by Part of Speech and Normalize
    for word in words:
        parsed = morph.parse(word)[0]
        pos = parsed.tag.POS
        
        # Keep only Nouns and Verbs (including infinitives)
        if pos in ['NOUN', 'VERB', 'INFN']:
            cleaned_words.append(parsed.normal_form)
            
    return cleaned_words

# --- Comprehensive Edge-Case Testing ---
dirty_text = """
В 2026 году бизнес-план компании "Окна-Маркет" по-прежнему лежал на столе.
Кто-то сказал: "Это же рок-н-ролл!" — и начал быстро-быстро писать код.
Из-за форс-мажора пришлось всё переделывать.
"""

result = clean_and_normalize_russian_text_advanced(dirty_text)
print(result)


In [ ]:
%timeit clean_and_normalize_russian_text_advanced(dirty_text)


In [ ]:
from timeit import timeit


timeit('clean_and_normalize_russian_text_advanced(dirty_text)', globals=globals(), number=1)


In [ ]:
small_df = df.head(10)

res = df.with_columns(
    pl.col('text').map_elements(clean_and_normalize_russian_text_advanced, return_dtype=pl.List(pl.String)).alias('clean_text')
).select(pl.col('clean_text')).explode('clean_text', empty_as_null=True, keep_nulls=False).group_by('clean_text').agg(pl.len())
res.sink_csv(DATA_DIR / 'processed' / 'words_frequency.csv')


In [ ]:
res.to_pandas()
